# Phase 2 CLEF CDAN+E Kaggle Runner

This notebook is a thin Kaggle driver for the repository scripts. It does not duplicate CDAN training logic.

Expected Kaggle inputs:

- A copy of this repository, or upload the repository as a Kaggle dataset.
- Phase 2 processed data files: `mitbih_train.npz`, `mitbih_test.npz`, `incart_unlabeled.npz`, `incart_test_heldout.npz`.
- CLEF checkpoint: `clef_small.ckpt`.
- Optional source-only CLEF fine-tune checkpoint: `clef_finetune_best.pt`. If it is missing, run the source-only CLEF notebook/script first or point `SOURCE_INIT_CHECKPOINT` to an uploaded checkpoint.

Outputs are written under `/kaggle/working/ecg_thesis/outputs/phase2_clef_cdan` and zipped to `/kaggle/working/phase2_clef_cdan_outputs.zip`.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT = Path('/kaggle/input')

# Update these paths to match your Kaggle dataset names.
REPO_INPUT_DIR = None  # Example: KAGGLE_INPUT / 'ecg-thesis-repo' / 'ecg_thesis'
DATA_INPUT_DIR = KAGGLE_INPUT / 'ecg-thesis-phase2-data'
CLEF_INPUT_CHECKPOINT = KAGGLE_INPUT / 'clef-small' / 'clef_small.ckpt'
SOURCE_INIT_CHECKPOINT = KAGGLE_INPUT / 'phase2-clef-source' / 'clef_finetune_best.pt'

PROJECT_DIR = KAGGLE_WORKING / 'ecg_thesis'
CONFIG_PATH = 'configs/phase2_clef_cdan.yaml'

RUN_SMOKE = True
RUN_FULL_TRAIN = False
RUN_EVAL = True
FULL_PREFIX = 'clef_cdan_e_from_finetune'
SMOKE_PREFIX = 'clef_cdan_e_smoke'
FULL_EPOCHS = None  # Keep None to use the config value.
SMOKE_SAMPLES = 32


## Locate Or Copy Repository

If this notebook is already inside a checked-out repo, the cell uses it. Otherwise, set `REPO_INPUT_DIR` above to the Kaggle input path containing `ecg_thesis`.

In [ ]:
def run(cmd, cwd=PROJECT_DIR):
    print('+', ' '.join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=str(cwd), check=True)

if not PROJECT_DIR.exists():
    here = Path.cwd()
    if (here / 'configs' / 'phase2_clef_cdan.yaml').exists() and (here / 'src').exists():
        PROJECT_DIR = here
    elif REPO_INPUT_DIR is not None and Path(REPO_INPUT_DIR).exists():
        shutil.copytree(REPO_INPUT_DIR, PROJECT_DIR)
    else:
        raise FileNotFoundError('Set REPO_INPUT_DIR to the Kaggle dataset path containing ecg_thesis.')

print('PROJECT_DIR =', PROJECT_DIR)
assert (PROJECT_DIR / CONFIG_PATH).exists(), CONFIG_PATH


## Install Dependencies

In [ ]:
requirements = PROJECT_DIR / 'requirements.txt'
if requirements.exists():
    run(['python', '-m', 'pip', 'install', '-q', '-r', str(requirements)], cwd=PROJECT_DIR)


## Prepare Data And Checkpoints

The repo config expects data under `data/processed` and the CLEF checkpoint under `models/clef/clef_small.ckpt`.

In [ ]:
processed_dir = PROJECT_DIR / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
for name in ['mitbih_train.npz', 'mitbih_test.npz', 'incart_unlabeled.npz', 'incart_test_heldout.npz']:
    dst = processed_dir / name
    if not dst.exists():
        src = DATA_INPUT_DIR / name
        if not src.exists():
            raise FileNotFoundError(f'Missing data file: {src}')
        shutil.copy2(src, dst)

clef_dir = PROJECT_DIR / 'models' / 'clef'
clef_dir.mkdir(parents=True, exist_ok=True)
clef_dst = clef_dir / 'clef_small.ckpt'
if not clef_dst.exists():
    if not CLEF_INPUT_CHECKPOINT.exists():
        raise FileNotFoundError(f'Missing CLEF checkpoint: {CLEF_INPUT_CHECKPOINT}')
    shutil.copy2(CLEF_INPUT_CHECKPOINT, clef_dst)

source_ckpt_dst = PROJECT_DIR / 'outputs' / 'phase2_clef' / 'checkpoints' / 'clef_finetune_best.pt'
if not source_ckpt_dst.exists():
    if not SOURCE_INIT_CHECKPOINT.exists():
        raise FileNotFoundError(f'Missing source init checkpoint: {SOURCE_INIT_CHECKPOINT}')
    source_ckpt_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(SOURCE_INIT_CHECKPOINT, source_ckpt_dst)

print('Prepared data and checkpoints.')


## Static Check

In [ ]:
run(['python', 'scripts/check_repo.py'])


## Smoke Run

In [ ]:
if RUN_SMOKE:
    run([
        'python', 'scripts/phase2_clef/08_train_cdan_clef.py',
        '--config', CONFIG_PATH,
        '--epochs', '1',
        '--max-source-samples', str(SMOKE_SAMPLES),
        '--max-target-samples', str(SMOKE_SAMPLES),
        '--max-val-samples', str(SMOKE_SAMPLES),
        '--checkpoint-prefix', SMOKE_PREFIX
    ])


## Full Training

In [ ]:
if RUN_FULL_TRAIN:
    cmd = ['python', 'scripts/phase2_clef/08_train_cdan_clef.py', '--config', CONFIG_PATH]
    if FULL_EPOCHS is not None:
        cmd += ['--epochs', str(FULL_EPOCHS)]
    run(cmd)


## Evaluation

In [ ]:
if RUN_EVAL:
    prefix = FULL_PREFIX if RUN_FULL_TRAIN else SMOKE_PREFIX
    checkpoint = PROJECT_DIR / 'outputs' / 'phase2_clef_cdan' / 'checkpoints' / f'{prefix}_best.pt'
    if not checkpoint.exists():
        raise FileNotFoundError(f'Missing CDAN checkpoint for eval: {checkpoint}')
    run([
        'python', 'scripts/phase2_clef/09_eval_cdan_clef.py',
        '--config', CONFIG_PATH,
        '--checkpoint', str(checkpoint),
        '--dataset', 'both',
        '--checkpoint-prefix', prefix
    ])
    run(['python', 'scripts/phase2_clef/10_make_cdan_report.py', '--config', CONFIG_PATH, '--checkpoint-prefix', prefix])


## Zip Outputs

In [ ]:
output_dir = PROJECT_DIR / 'outputs' / 'phase2_clef_cdan'
zip_base = KAGGLE_WORKING / 'phase2_clef_cdan_outputs'
if output_dir.exists():
    archive = shutil.make_archive(str(zip_base), 'zip', root_dir=str(output_dir))
    print('Wrote', archive)
else:
    print('No output directory found:', output_dir)
